In [ ]:
# imports
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import os
import numpy as np

In [ ]:


PROJECT_DIR = Path(__file__).resolve().parents[1]
DATA_DIR = PROJECT_DIR / "data"
FIGURES_DIR = PROJECT_DIR / "figures"


def import_data():
    csv_paths = sorted(DATA_DIR.glob("*.csv"))
    df_list = [pd.read_csv(path) for path in csv_paths]
    df = pd.concat(df_list, ignore_index=True)
    return df


def clean_data(df):
    clean_df = df.copy()
    clean_df.columns = [str(col).strip() for col in clean_df.columns]

    if "easy_hard" in clean_df.columns:
        if "word" in clean_df.columns:
            clean_df["word"] = clean_df["word"].combine_first(
                clean_df["easy_hard"]
            )
            clean_df = clean_df.drop(columns=["easy_hard"])
        else:
            clean_df = clean_df.rename(columns={"easy_hard": "word"})
    if "old_new" in clean_df.columns:
        if "easy_hard" in clean_df.columns:
            clean_df["easy_hard"] = clean_df["easy_hard"].combine_first(
                clean_df["old_new"]
            )
            clean_df = clean_df.drop(columns=["old_new"])
        else:
            clean_df = clean_df.rename(columns={"old_new": "easy_hard"})

    required_cols = [
        "word",
        "easy_hard",
        "old_new",
        "R_F",
        "testing_input.rt",
    ]
    missing = [col for col in required_cols if col not in clean_df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    clean_df["word"] = clean_df["word"].astype(str)
    clean_df["easy_hard"] = clean_df["easy_hard"].astype(str)
    clean_df["old_new"] = clean_df["old_new"].astype(str)
    clean_df["R_F"] = clean_df["R_F"].astype(str)
    clean_df["testing_input.rt"] = clean_df["testing_input.rt"].astype(str)



    stim_raw = clean_df["easy_hard"].astype(str).str.lower().str.strip()
    stim_map = {"0": "new", "1": "old", "new": "new", "old": "old"}
    clean_df["easy_hard"] = stim_raw.map(stim_map).fillna(stim_raw)

    exp_map = {
        "0": "words",
        "1": "images",
        "text": "words",
        "word": "words",
        "picture": "images",
        "image": "images",
    }
    clean_df["word"] = clean_df["word"].replace(exp_map)
    return clean_df


def create_participant_histograms(trimmed_df):
    participant_cond_df = get_summarized_means(trimmed_df, "word")
    participant_means = participant_cond_df.groupby(
        "participant_id", as_index=False
    )

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    axes[0].hist(
        participant_means["old_new"], bins=12, color="#DF7BDD", edgecolor="black"
    )
    axes[0].set_title("Old Vs New Remember Accuracy")
    axes[0].set_xlabel("Old vs New")
    axes[0].set_ylabel("Remember Accuracy")

    axes[1].hist(
        participant_means["R_F"], bins=12, color="#8E3EE3", edgecolor="black"
    )
    axes[1].set_title("Remember Vs Forget Remember Accuracy")
    axes[1].set_xlabel("Remember Vs Forget")
    axes[1].set_ylabel("Remember Accuracy")

    fig.suptitle("Participant-Level Performance Distributions", fontsize=14)
    plt.tight_layout()
    FIGURES_DIR.mkdir(parents=True, exist_ok=True)
    fig.savefig(FIGURES_DIR / "participant_histograms.png", dpi=300)
    plt.close(fig)

def main():
    df = import_data()
    clean_df = clean_data(df)

    create_participant_histograms(trimmed_df)


    print("Analysis complete.")
    print(f"Rows loaded: {len(df)}")
    print(f"Rows after cleaning: {len(clean_df)}")


if __name__ == "__main__":
    main()
